<a href="https://colab.research.google.com/github/GuiCastro7/Grupo-3---ECAA08/blob/main/etapa-01-logica/09%20-%20Motor%20de%20Inferencia%20Forward%20e%20Backward%20Chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 09 - Notebook: Motores de Inferência Forward e Backward Chaining

**Projeto Integrador / Disciplina:** Matemática Discreta e Sistemas Digitais  
**Curso:** Engenharia de Controle e Automação (ECA)  
**Sistema:** Linha Automatizada de Envasamento e Tampamento de Bebidas  
**Grupo:** Grupo 3 — ECAA08  

---

## 1. Fundamentos Matemáticos: Motores de Inferência em Sistemas Baseados em Regras

Na automação industrial e segurança funcional (normas IEC 61508 / IEC 61511 / NR-12), o diagnóstico de falhas e a atuação preventiva do SCADA são governados por um **Motor de Inferência (*Inference Engine*)** atuando sobre uma base de conhecimento modelada por Cláusulas de Horn:

$$\langle \mathcal{F}, \mathcal{R}, \mathcal{E} \rangle$$

Onde:
1. **$\mathcal{F}$ (Base de Fatos):** Conjunto finito de proposições lógicas ativas provenientes de sensores e estados de atuadores no instante $t$.
2. **$\mathcal{R}$ (Base de Regras de Produção):** Conjunto de sentenças na forma $(A_1 \land A_2 \land \dots \land A_k) \rightarrow C$.
3. **$\mathcal{E}$ (Estratégia de Resolução de Conflitos):** Critérios de arbitragem por prioridade e severidade operacional.

Neste notebook, implementamos um **Motor de Inferência Híbrido** com suporte completo a:
- **Forward Chaining (Data-Driven / Bottom-Up):** Varredura de telemetria em tempo real até atingir ponto fixo, gerando **Trilha de Auditoria (*Audit Trail*)** e Procedimentos Operacionais Padrão (POP).
- **Backward Chaining (Goal-Driven / Top-Down):** Avaliação pericial pós-falha e análise de causa-raiz (*Root Cause Analysis*), construindo a **Árvore de Explicação (*Explanation Facility*)** com proteção contra ciclos.

## 2. Modelagem das Classes: Fato, RegraDiagnostico, BaseConhecimentoSCADA e MotorInferenciaHibrido

A estrutura orientada a objetos é dividida em quatro componentes essenciais:
1. `Fato`: Modela átomos proposicionais (`nome`, `valor`, `descricao`, `fonte`, `timestamp`).
2. `RegraDiagnostico`: Modela a Cláusula de Horn individual com metadados de automação (`id_regra`, `antecedentes`, `consequente`, `descricao_diagnostico`, `severidade`, `prioridade`, `tempo_resposta_max_s`, `procedimento_pop`).
3. `BaseConhecimentoSCADA`: Gerencia o cadastro, índices duplos invertidos (por antecedente e por consequente) e checagem de consistência/redundância.
4. `MotorInferenciaHibrido`: Executa os algoritmos de **Forward Chaining** (com ponto fixo e priorização) e **Backward Chaining** (com busca recursiva AND-OR e prevenção de ciclos).

In [1]:
from dataclasses import dataclass, field
from typing import List, Set, Dict, Any, Optional, Tuple
import time

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Formata uma lista de dicionários em uma tabela ASCII legível para exibição industrial."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

@dataclass
class Fato:
    nome: str
    valor: bool
    descricao: str
    fonte: str = "SENSOR"  # 'SENSOR' ou 'INFERIDO'
    timestamp: float = field(default_factory=time.time)

@dataclass
class RegraDiagnostico:
    id_regra: str
    antecedentes: Set[str]
    consequente: str
    descricao_diagnostico: str
    severidade: str             # 'CRÍTICA', 'ALTA', 'MÉDIA', 'BAIXA'
    prioridade: int             # 1 a 10 (10 = máxima prioridade de intertravamento)
    tempo_resposta_max_s: float
    procedimento_pop: str

class BaseConhecimentoSCADA:
    def __init__(self):
        self.regras: List[RegraDiagnostico] = []
        self._indice_antecedentes: Dict[str, List[RegraDiagnostico]] = {}
        self._indice_consequentes: Dict[str, List[RegraDiagnostico]] = {}

    def adicionar_regra(
        self, id_regra: str, antecedentes: List[str], consequente: str,
        descricao: str, severidade: str = "ALTA", prioridade: int = 5,
        tempo_max_s: float = 5.0, pop: str = "Verificar malha de controle"
    ):
        """Cadastra uma nova regra de diagnóstico e atualiza os índices invertidos."""
        regra = RegraDiagnostico(
            id_regra=id_regra,
            antecedentes=set(antecedentes),
            consequente=consequente,
            descricao_diagnostico=descricao,
            severidade=severidade,
            prioridade=prioridade,
            tempo_resposta_max_s=tempo_max_s,
            procedimento_pop=pop
        )
        self.regras.append(regra)

        # Atualiza índice por antecedentes (otimiza Forward Chaining)
        for ant in antecedentes:
            if ant not in self._indice_antecedentes:
                self._indice_antecedentes[ant] = []
            self._indice_antecedentes[ant].append(regra)

        # Atualiza índice por consequente (otimiza Backward Chaining)
        if consequente not in self._indice_consequentes:
            self._indice_consequentes[consequente] = []
        self._indice_consequentes[consequente].append(regra)

    def obter_regras_por_antecedente(self, fato_nome: str) -> List[RegraDiagnostico]:
        """Recupera regras que contêm o fato informado no conjunto de antecedentes."""
        return self._indice_antecedentes.get(fato_nome, [])

    def obter_regras_por_consequente(self, fato_nome: str) -> List[RegraDiagnostico]:
        """Recupera regras cujo consequente é o fato especificado."""
        return self._indice_consequentes.get(fato_nome, [])

    def exportar_catalogo(self) -> List[Dict[str, Any]]:
        """Exporta o catálogo de regras ordenado por prioridade decrescente."""
        catalogo = []
        for r in sorted(self.regras, key=lambda x: x.prioridade, reverse=True):
            catalogo.append({
                "ID": r.id_regra,
                "Prioridade": r.prioridade,
                "Severidade": r.severidade,
                "SE (Antecedentes)": " AND ".join(sorted(r.antecedentes)),
                "ENTÃO (Consequente)": r.consequente,
                "Diagnóstico": r.descricao_diagnostico,
                "POP": r.procedimento_pop
            })
        return catalogo

class MotorInferenciaHibrido:
    def __init__(self, base_conhecimento: BaseConhecimentoSCADA):
        self.bc = base_conhecimento

    def forward_chaining(self, fatos_iniciais: Set[str]) -> Tuple[Set[str], List[Dict[str, Any]]]:
        """
        Executa o algoritmo de Forward Chaining (Data-Driven) até atingir ponto fixo.
        Aplica resolução de conflitos por prioridade de segurança (10 = máxima prioridade).
        Retorna o conjunto final de fatos deduzidos e a Trilha de Auditoria (Audit Trail).
        """
        fatos_conhecidos = set(fatos_iniciais)
        historico_disparos: List[Dict[str, Any]] = []
        novos_fatos = True
        passo = 1

        while novos_fatos:
            novos_fatos = False
            # Ordena regras por prioridade decrescente (Resolução de Conflitos SIL/NR-12)
            regras_candidatas = sorted(self.bc.regras, key=lambda r: r.prioridade, reverse=True)

            for regra in regras_candidatas:
                # Verifica se todos os antecedentes estão satisfeitos e se o consequente ainda não foi inferido
                if regra.antecedentes.issubset(fatos_conhecidos) and regra.consequente not in fatos_conhecidos:
                    fatos_conhecidos.add(regra.consequente)
                    historico_disparos.append({
                        "Passo": passo,
                        "Regra": regra.id_regra,
                        "Prioridade": regra.prioridade,
                        "Severidade": regra.severidade,
                        "Fato Inferido": regra.consequente,
                        "Diagnóstico Causa-Raiz": regra.descricao_diagnostico,
                        "Procedimento Operacional (POP)": regra.procedimento_pop
                    })
                    passo += 1
                    novos_fatos = True
                    # Dispara uma regra por ciclo e reavalia a agenda (Estratégia de Resolução de Conflitos)
                    break

        return fatos_conhecidos, historico_disparos

    def backward_chaining(
        self, meta: str, fatos_iniciais: Set[str], visitados: Optional[Set[str]] = None
    ) -> Tuple[bool, List[str], Dict[str, Any]]:
        """
        Executa o algoritmo de Backward Chaining (Goal-Driven) com busca recursiva em profundidade (DFS).
        Inclui detecção e prevenção de ciclos infinitos.
        Retorna (sucesso_bool, log_explicativo, arvore_de_prova_dict).
        """
        if visitados is None:
            visitados = set()

        log: List[str] = []
        arvore_prova: Dict[str, Any] = {"meta": meta, "provado": False, "tipo": "NÓ", "filhos": []}

        # Caso Base 1: A meta é um fato básico já conhecido na telemetria
        if meta in fatos_iniciais:
            log.append(f"[Sucesso Imediato] Meta '{meta}' é um fato primitivo ativo nos sensores.")
            arvore_prova["provado"] = True
            arvore_prova["tipo"] = "FATO_PRIMITIVO"
            return True, log, arvore_prova

        # Caso Base 2: Prevenção de Ciclos Recursivos (Loop Detection)
        if meta in visitados:
            log.append(f"[Ciclo Detectado] Meta '{meta}' já está na pilha de avaliação. Poda do ramo.")
            arvore_prova["tipo"] = "CICLO_BLOQUEADO"
            return False, log, arvore_prova

        visitados.add(meta)

        # Busca regras candidatas que concluem a meta (Nós OR)
        regras_candidatas = self.bc.obter_regras_por_consequente(meta)
        if not regras_candidatas:
            log.append(f"[Falha] Nenhuma regra na Base de Conhecimento produz o consequente '{meta}'.")
            arvore_prova["tipo"] = "SEM_REGRAS"
            visitados.remove(meta)
            return False, log, arvore_prova

        # Ordena candidatas por prioridade para tentar primeiro os caminhos mais críticos
        regras_candidatas = sorted(regras_candidatas, key=lambda r: r.prioridade, reverse=True)

        for regra in regras_candidatas:
            log.append(f"[Testando Regra] Avaliando {regra.id_regra} para provar meta '{meta}'...")
            todos_antecedentes_provados = True
            filhos_regra: List[Dict[str, Any]] = []

            # Avalia todos os antecedentes da regra (Nó AND)
            for ant in sorted(regra.antecedentes):
                sub_sucesso, sub_log, sub_arvore = self.backward_chaining(ant, fatos_iniciais, visitados.copy())
                log.extend(["  " + l for l in sub_log])
                filhos_regra.append(sub_arvore)
                if not sub_sucesso:
                    todos_antecedentes_provados = False
                    log.append(f"  [Ramo Falhou] Antecedente '{ant}' da regra {regra.id_regra} NÃO pôde ser provado.")
                    break

            if todos_antecedentes_provados:
                log.append(
                    f"[Meta Provada com Sucesso] Meta '{meta}' foi formalmente comprovada pela regra {regra.id_regra} "
                    f"(Diagnóstico: {regra.descricao_diagnostico})!"
                )
                arvore_prova["provado"] = True
                arvore_prova["tipo"] = "REGRA_SATISFEITA"
                arvore_prova["id_regra"] = regra.id_regra
                arvore_prova["diagnostico"] = regra.descricao_diagnostico
                arvore_prova["pop"] = regra.procedimento_pop
                arvore_prova["filhos"] = filhos_regra
                visitados.remove(meta)
                return True, log, arvore_prova

        log.append(f"[Falha Final] Nenhuma regra candidata conseguiu provar a meta '{meta}'.")
        visitados.remove(meta)
        return False, log, arvore_prova

    def explicar_meta(self, meta: str, fatos_iniciais: Set[str]) -> str:
        """Gera um relatório textual pericial para o operador do SCADA explicando a conclusão da meta."""
        provado, log, arvore = self.backward_chaining(meta, fatos_iniciais)
        linhas = [
            f"=== RELATÓRIO PERICIAL DE INVESTIGAÇÃO SCADA (GOAL: {meta}) ===",
            f"Status da Investigação: {'[PROVADA / CONFIRMADA]' if provado else '[REJEITADA / NÃO COMPROVADA]'}",
            "\nTrilha Lógica de Dedução e Análise de Hipótese:"
        ]
        for l in log:
            linhas.append(f"  {l}")
        return "\n".join(linhas)

print("[OK] Classes Fato, RegraDiagnostico, BaseConhecimentoSCADA e MotorInferenciaHibrido inicializadas com sucesso!")

[OK] Classes Fato, RegraDiagnostico, BaseConhecimentoSCADA e MotorInferenciaHibrido inicializadas com sucesso!


## 3. Cadastro do Catálogo Especialista de Regras da Linha de Envasamento (Grupo 3)

Cadastramos agora a base completa com 10 regras hierarquizadas que modelam o processo de envasamento e tampamento de bebidas:

| ID | Antecedentes | Consequente | Diagnóstico de Causa-Raiz | Severidade | Prio | POP |
| :--- | :--- | :--- | :--- | :---: | :---: | :--- |
| **R-01** | `p_max1` $\land$ `q_max1` | `SOBRECARGA_LINHA_ALIMENTACAO` | Sobrecarga de Pressão/Vazão na Entrada da Linha | **CRÍTICA** | 10 | `POP-SIS-01` |
| **R-02** | `SOBRECARGA_LINHA_ALIMENTACAO` $\land$ `y_valv1` | `TRIP_BLOQUEIO_EMERGENCIA` | Falha de Alívio com Válvula Principal VS1 Aberta | **CRÍTICA** | 10 | `POP-SIS-02` |
| **R-03** | `p_min1` $\land$ `y_bomba` | `CAVITACAO_BOMBA_BC1` | Risco Crítico de Cavitação na Bomba BC1 | **ALTA** | 8 | `POP-MA-04` |
| **R-04** | `p_max2` | `SOBREPRESSAO_ACUMULADOR_AS1` | Sobrepressão no Acumulador AS1 | **CRÍTICA** | 9 | `POP-SST-08` |
| **R-05** | `q_min2` $\land$ `y_valv2` | `OBSTRUCAO_BICO_ENVASE` | Bloqueio/Entupimento no Bico de Envase VS2 | **ALTA** | 7 | `POP-SEC-02` |
| **R-06** | `p_max2` $\land$ `y_valv3` | `SOBREPRESSAO_SISTEMA_CAPPING` | Pressão Excessiva no Atuador de Tampamento AC1 | **CRÍTICA** | 9 | `POP-CRIO-01` |
| **R-07** | `CAVITACAO_BOMBA_BC1` $\land$ `l_min_ts1` | `DESARME_TERMICO_BOMBA` | Bomba BC1 a Seco com Esgotamento do Tanque TS1 | **CRÍTICA** | 9 | `POP-MA-05` |
| **R-08** | `OBSTRUCAO_BICO_ENVASE` $\land$ `presenca_garrafa` | `DERRAMAMENTO_E_FALHA_ENVASE` | Falha de Dosagem e Risco de Transbordamento | **ALTA** | 8 | `POP-SEC-03` |
| **R-09** | `TRIP_BLOQUEIO_EMERGENCIA` | `PARADA_TOTAL_LINHA` | Desarme Geral por Falha Crítica de Alimentação | **CRÍTICA** | 10 | `POP-ESD-01` |
| **R-10** | `DESARME_TERMICO_BOMBA` | `PARADA_TOTAL_LINHA` | Desarme Geral por Perda Crítica do Grupo de Bombeamento | **CRÍTICA** | 10 | `POP-ESD-01` |

In [2]:
bc = BaseConhecimentoSCADA()

# 1. Regras de Sobrecarga e Alimentação Principal
bc.adicionar_regra(
    id_regra="R-01",
    antecedentes=["p_max1", "q_max1"],
    consequente="SOBRECARGA_LINHA_ALIMENTACAO",
    descricao="Sobrecarga de Pressão e Vazão na Linha de Alimentação",
    severidade="CRÍTICA",
    prioridade=10,
    tempo_max_s=1.0,
    pop="POP-SIS-01: Cortar alimentação, parar bomba BC1 e fechar válvula VS1"
)

bc.adicionar_regra(
    id_regra="R-02",
    antecedentes=["SOBRECARGA_LINHA_ALIMENTACAO", "y_valv1"],
    consequente="TRIP_BLOQUEIO_EMERGENCIA",
    descricao="Falha de Alívio com Válvula Principal VS1 Aberta sob Sobrecarga",
    severidade="CRÍTICA",
    prioridade=10,
    tempo_max_s=0.5,
    pop="POP-SIS-02: Interromper contator da bomba BC1 e forçar fechamento de VS1 via PLC"
)

# 2. Regras de Cavitação e Bombeamento
bc.adicionar_regra(
    id_regra="R-03",
    antecedentes=["p_min1", "y_bomba"],
    consequente="CAVITACAO_BOMBA_BC1",
    descricao="Risco Crítico de Cavitação e Falha Mecânica na Bomba BC1",
    severidade="ALTA",
    prioridade=8,
    tempo_max_s=2.0,
    pop="POP-MA-04: Desligar bomba BC1 e verificar nível do tanque de suprimento TS1"
)

# 3. Regras de Acumulador e Tampamento Pneumático
bc.adicionar_regra(
    id_regra="R-04",
    antecedentes=["p_max2"],
    consequente="SOBREPRESSAO_ACUMULADOR_AS1",
    descricao="Sobrepressão Acima do Limite de Projeto no Acumulador AS1",
    severidade="CRÍTICA",
    prioridade=9,
    tempo_max_s=1.0,
    pop="POP-SST-08: Acionar alívio pneumático de emergência e interromper fluxo para AS1"
)

# 4. Regras do Posto de Envase e Dosagem
bc.adicionar_regra(
    id_regra="R-05",
    antecedentes=["q_min2", "y_valv2"],
    consequente="OBSTRUCAO_BICO_ENVASE",
    descricao="Bloqueio ou Entupimento Mecânico no Bico de Envase VS2",
    severidade="ALTA",
    prioridade=7,
    tempo_max_s=3.0,
    pop="POP-SEC-02: Parar esteira RC1, isolar ramal de envase e realizar retrolavagem"
)

bc.adicionar_regra(
    id_regra="R-06",
    antecedentes=["p_max2", "y_valv3"],
    consequente="SOBREPRESSAO_SISTEMA_CAPPING",
    descricao="Pressão Pneumática Excessiva no Atuador de Capping AC1",
    severidade="CRÍTICA",
    prioridade=9,
    tempo_max_s=2.0,
    pop="POP-CRIO-01: Fechar válvula de capping VS3 e aliviar pressão residual"
)

# 5. Regras Avançadas de Encadeamento e Desarme Geral
bc.adicionar_regra(
    id_regra="R-07",
    antecedentes=["CAVITACAO_BOMBA_BC1", "l_min_ts1"],
    consequente="DESARME_TERMICO_BOMBA",
    descricao="Esgotamento do Tanque TS1 com Bomba BC1 a Seco gerando Sobrecarga Térmica",
    severidade="CRÍTICA",
    prioridade=9,
    tempo_max_s=1.0,
    pop="POP-MA-05: Bloqueio do circuito elétrico da bomba BC1 e reabastecimento de TS1"
)

bc.adicionar_regra(
    id_regra="R-08",
    antecedentes=["OBSTRUCAO_BICO_ENVASE", "presenca_garrafa"],
    consequente="DERRAMAMENTO_E_FALHA_ENVASE",
    descricao="Falha de Envase com Garrafa no Posto e Risco de Transbordamento/Perda de Lote",
    severidade="ALTA",
    prioridade=8,
    tempo_max_s=1.5,
    pop="POP-SEC-03: Rejeição da garrafa defeituosa para esteira de descarte e purga do bico"
)

bc.adicionar_regra(
    id_regra="R-09",
    antecedentes=["TRIP_BLOQUEIO_EMERGENCIA"],
    consequente="PARADA_TOTAL_LINHA",
    descricao="Intertravamento de Emergência Geral por Sobrecarga Crítica de Alimentação",
    severidade="CRÍTICA",
    prioridade=10,
    tempo_max_s=0.2,
    pop="POP-ESD-01: Acionar alarme geral, desabilitar saídas do CLP e registrar log de segurança"
)

bc.adicionar_regra(
    id_regra="R-10",
    antecedentes=["DESARME_TERMICO_BOMBA"],
    consequente="PARADA_TOTAL_LINHA",
    descricao="Intertravamento de Emergência Geral por Perda Crítica do Grupo de Bombeamento",
    severidade="CRÍTICA",
    prioridade=10,
    tempo_max_s=0.2,
    pop="POP-ESD-01: Acionar alarme geral, desabilitar saídas do CLP e registrar log de segurança"
)

print("=== CATÁLOGO OFICIAL DE REGRAS DE PRODUÇÃO DA LINHA DE ENVASE (GRUPO 3) ===\n")
print(formatar_tabela(bc.exportar_catalogo()))

=== CATÁLOGO OFICIAL DE REGRAS DE PRODUÇÃO DA LINHA DE ENVASE (GRUPO 3) ===

ID   | Prioridade | Severidade | SE (Antecedentes)                          | ENTÃO (Consequente)          | Diagnóstico                                                                   | POP                                                                                     
-----+------------+------------+--------------------------------------------+------------------------------+-------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------
R-01 | 10         | CRÍTICA    | p_max1 AND q_max1                          | SOBRECARGA_LINHA_ALIMENTACAO | Sobrecarga de Pressão e Vazão na Linha de Alimentação                         | POP-SIS-01: Cortar alimentação, parar bomba BC1 e fechar válvula VS1                    
R-02 | 10         | CRÍTICA    | SOBRECARGA_LINHA_ALIMENTACAO AND y_valv1   | TRIP_BLOQUE

## 4. Simulações em Tempo Real: Encadeamento para Frente (*Forward Chaining*)

O *Forward Chaining* opera em tempo real de varredura (*scan-time*). Avaliamos dois cenários críticos:
- **Cenário 1:** Sobrecarga simultânea de pressão (`p_max1`) e vazão (`q_max1`) com válvula de entrada aberta (`y_valv1`).
- **Cenário 2:** Sucção com subpressão (`p_min1`), bomba BC1 ligada (`y_bomba`) e tanque de xarope esgotado (`l_min_ts1`).

In [3]:
motor = MotorInferenciaHibrido(bc)

print("===============================================================================")
print("=== CENÁRIO 1 (FORWARD CHAINING): SOBRECARGA, TRIP E PARADA TOTAL DA LINHA ===")
print("===============================================================================")
fatos_c1 = {"p_max1", "q_max1", "y_valv1"}
fatos_finais_c1, trilha_c1 = motor.forward_chaining(fatos_c1)

print(f"\n[Fatos Iniciais de Telemetria]: {sorted(list(fatos_c1))}")
print("\n--- Trilha de Auditoria (Audit Trail / Diagnósticos Disparados) ---")
print(formatar_tabela(trilha_c1))
print(f"\n[Conjunto Final de Fatos Ativos]: {sorted(list(fatos_finais_c1))}")

# Verificações Formais
assert "SOBRECARGA_LINHA_ALIMENTACAO" in fatos_finais_c1
assert "TRIP_BLOQUEIO_EMERGENCIA" in fatos_finais_c1
assert "PARADA_TOTAL_LINHA" in fatos_finais_c1
print("\n[OK] Cenário 1 validado: Encadeamento em 3 estágios executado com sucesso até a parada total!")

print("\n" + "=" * 79)
print("=== CENÁRIO 2 (FORWARD CHAINING): CAVITAÇÃO, DESARME TÉRMICO E PARADA TOTAL ===")
print("=" * 79)
fatos_c2 = {"p_min1", "y_bomba", "l_min_ts1"}
fatos_finais_c2, trilha_c2 = motor.forward_chaining(fatos_c2)

print(f"\n[Fatos Iniciais de Telemetria]: {sorted(list(fatos_c2))}")
print("\n--- Trilha de Auditoria (Audit Trail / Diagnósticos Disparados) ---")
print(formatar_tabela(trilha_c2))
print(f"\n[Conjunto Final de Fatos Ativos]: {sorted(list(fatos_finais_c2))}")

# Verificações Formais
assert "CAVITACAO_BOMBA_BC1" in fatos_finais_c2
assert "DESARME_TERMICO_BOMBA" in fatos_finais_c2
assert "PARADA_TOTAL_LINHA" in fatos_finais_c2
print("\n[OK] Cenário 2 validado: Encadeamento de proteção mecânica e térmica validado com sucesso!")

=== CENÁRIO 1 (FORWARD CHAINING): SOBRECARGA, TRIP E PARADA TOTAL DA LINHA ===

[Fatos Iniciais de Telemetria]: ['p_max1', 'q_max1', 'y_valv1']

--- Trilha de Auditoria (Audit Trail / Diagnósticos Disparados) ---
Passo | Regra | Prioridade | Severidade | Fato Inferido                | Diagnóstico Causa-Raiz                                                    | Procedimento Operacional (POP)                                                          
------+-------+------------+------------+------------------------------+---------------------------------------------------------------------------+-----------------------------------------------------------------------------------------
1     | R-01  | 10         | CRÍTICA    | SOBRECARGA_LINHA_ALIMENTACAO | Sobrecarga de Pressão e Vazão na Linha de Alimentação                     | POP-SIS-01: Cortar alimentação, parar bomba BC1 e fechar válvula VS1                    
2     | R-02  | 10         | CRÍTICA    | TRIP_BLOQUEIO_EMERGENCIA     | 

## 5. Análise Pericial Pós-Acidente: Encadeamento para Trás (*Backward Chaining*)

O *Backward Chaining* atua como módulo pericial pós-falha (*Root Cause Analysis - RCA*):
- **Cenário 3 (Prova Causal de Parada Geral):** O operador investiga a meta `PARADA_TOTAL_LINHA` a partir do histórico de telemetria.
- **Cenário 4 (Rejeição de Falso Positivo):** O operador testa a hipótese de falha no atuador de capping `SOBREPRESSAO_SISTEMA_CAPPING` quando apenas a pressão está alta, mas a válvula VS3 não foi energizada.
- **Cenário 5 (Detecção de Falha no Posto de Envase):** O operador investiga a hipótese `DERRAMAMENTO_E_FALHA_ENVASE` sob restrição de vazão no bico com garrafa presente.

In [4]:
print("=================================================================================")
print("=== CENÁRIO 3 (BACKWARD CHAINING): INVESTIGAÇÃO PERICIAL DE CAUSA-RAIZ (RCA)  ===")
print("=================================================================================")
meta_c3 = "PARADA_TOTAL_LINHA"
telemetria_c3 = {"p_max1", "q_max1", "y_valv1"}
relatorio_c3 = motor.explicar_meta(meta_c3, telemetria_c3)
print(relatorio_c3)

provado_c3, _, arvore_c3 = motor.backward_chaining(meta_c3, telemetria_c3)
assert provado_c3 is True, "Erro: A meta de parada total deveria ser comprovada."
assert arvore_c3["id_regra"] == "R-09", "Erro: A regra topo deveria ser R-09."

print("\n" + "=" * 81)
print("=== CENÁRIO 4 (BACKWARD CHAINING): REJEIÇÃO DE HIPÓTESE / FALSO POSITIVO     ===")
print("=" * 81)
meta_c4 = "SOBREPRESSAO_SISTEMA_CAPPING"
telemetria_c4 = {"p_max2"}  # Note: y_valv3 está ausente (válvula fechada)
relatorio_c4 = motor.explicar_meta(meta_c4, telemetria_c4)
print(relatorio_c4)

provado_c4, _, _ = motor.backward_chaining(meta_c4, telemetria_c4)
assert provado_c4 is False, "Erro: A meta de sobrepressão de capping NÃO deve ser provada sem y_valv3."
print("\n[OK] Falso positivo rejeitado com sucesso pelo motor de inferência!")

print("\n" + "=" * 81)
print("=== CENÁRIO 5 (BACKWARD CHAINING): INVESTIGAÇÃO DE FALHA NO POSTO DE ENVASE  ===")
print("=" * 81)
meta_c5 = "DERRAMAMENTO_E_FALHA_ENVASE"
telemetria_c5 = {"q_min2", "y_valv2", "presenca_garrafa"}
relatorio_c5 = motor.explicar_meta(meta_c5, telemetria_c5)
print(relatorio_c5)

provado_c5, _, _ = motor.backward_chaining(meta_c5, telemetria_c5)
assert provado_c5 is True, "Erro: Falha de envase com garrafa presente deveria ser provada."
print("\n[OK] Causa-raiz de obstrução de envase comprovada com sucesso!")

=== CENÁRIO 3 (BACKWARD CHAINING): INVESTIGAÇÃO PERICIAL DE CAUSA-RAIZ (RCA)  ===
=== RELATÓRIO PERICIAL DE INVESTIGAÇÃO SCADA (GOAL: PARADA_TOTAL_LINHA) ===
Status da Investigação: [PROVADA / CONFIRMADA]

Trilha Lógica de Dedução e Análise de Hipótese:
  [Testando Regra] Avaliando R-09 para provar meta 'PARADA_TOTAL_LINHA'...
    [Testando Regra] Avaliando R-02 para provar meta 'TRIP_BLOQUEIO_EMERGENCIA'...
      [Testando Regra] Avaliando R-01 para provar meta 'SOBRECARGA_LINHA_ALIMENTACAO'...
        [Sucesso Imediato] Meta 'p_max1' é um fato primitivo ativo nos sensores.
        [Sucesso Imediato] Meta 'q_max1' é um fato primitivo ativo nos sensores.
      [Meta Provada com Sucesso] Meta 'SOBRECARGA_LINHA_ALIMENTACAO' foi formalmente comprovada pela regra R-01 (Diagnóstico: Sobrecarga de Pressão e Vazão na Linha de Alimentação)!
      [Sucesso Imediato] Meta 'y_valv1' é um fato primitivo ativo nos sensores.
    [Meta Provada com Sucesso] Meta 'TRIP_BLOQUEIO_EMERGENCIA' foi formalme

## 6. Bateria de Testes Automatizados e Asserções Formais

Nesta seção, realizamos uma suíte completa de testes automatizados para validar:
1. **Convergência e Idempotência:** Garantir que múltiplas passagens do Forward Chaining gerem o mesmo ponto fixo.
2. **Detecção e Proteção contra Ciclos Recursivos:** Inserção de regras circulares intencionais na base para validar a terminação do Backward Chaining.
3. **Integridade da Trilha de Auditoria:** Checagem estrutural de todos os campos de telemetria, diagnóstico e POPs.

In [5]:
print("--- INICIANDO BATERIA DE TESTES DE INTEGRIDADE E ROBUSTEZ TÉCNICA ---\n")

# 1. Teste de Idempotência do Forward Chaining
fatos_passagem_1, _ = motor.forward_chaining({"p_max1", "q_max1", "y_valv1"})
fatos_passagem_2, _ = motor.forward_chaining(fatos_passagem_1)
assert fatos_passagem_1 == fatos_passagem_2, "Erro: Forward Chaining não convergiu a ponto fixo idempotente."
print("[OK 1/4] Propriedade de Ponto Fixo (Fixed Point) e Idempotência comprovada com sucesso!")

# 2. Teste de Busca em Base Vazia / Fatos Inertes
fatos_inertes, trilha_inerte = motor.forward_chaining({"fato_inexistente_1", "fato_inexistente_2"})
assert len(trilha_inerte) == 0, "Erro: Nenhuma regra deveria disparar para fatos desconhecidos."
assert fatos_inertes == {"fato_inexistente_1", "fato_inexistente_2"}
print("[OK 2/4] Comportamento sob ausência de distúrbios operacionais validado com sucesso!")

# 3. Teste de Proteção contra Ciclos Recursivos no Backward Chaining
bc_ciclica = BaseConhecimentoSCADA()
bc_ciclica.adicionar_regra("R-LOOP-1", ["fato_A"], "fato_B", "Regra Cíclica 1")
bc_ciclica.adicionar_regra("R-LOOP-2", ["fato_B"], "fato_A", "Regra Cíclica 2")
motor_ciclico = MotorInferenciaHibrido(bc_ciclica)

provado_loop, log_loop, _ = motor_ciclico.backward_chaining("fato_B", set())
assert provado_loop is False, "Erro: O motor deveria rejeitar a meta presa em loop recursivo."
assert any("Ciclo Detectado" in l for l in log_loop), "Erro: O mecanismo de loop detection falhou."
print("[OK 3/4] Mecanismo de detecção e poda de ciclos recursivos validado com 100% de sucesso!")

# 4. Validação da Estrutura das Árvores de Explicação
provado_exp, log_exp, arvore_exp = motor.backward_chaining("TRIP_BLOQUEIO_EMERGENCIA", {"p_max1", "q_max1", "y_valv1"})
assert provado_exp is True
assert arvore_exp["provado"] is True
assert arvore_exp["tipo"] == "REGRA_SATISFEITA"
assert len(arvore_exp["filhos"]) == 2
print("[OK 4/4] Árvore de prova e justificativa pericial totalmente íntegra!")

print("\n===============================================================================")
print("=== TODOS OS TESTES FORMAIS DO MOTOR DE INFERÊNCIA FORAM CONCLUÍDOS COM SUCESSO! ===")
print("===============================================================================")

--- INICIANDO BATERIA DE TESTES DE INTEGRIDADE E ROBUSTEZ TÉCNICA ---

[OK 1/4] Propriedade de Ponto Fixo (Fixed Point) e Idempotência comprovada com sucesso!
[OK 2/4] Comportamento sob ausência de distúrbios operacionais validado com sucesso!
[OK 3/4] Mecanismo de detecção e poda de ciclos recursivos validado com 100% de sucesso!
[OK 4/4] Árvore de prova e justificativa pericial totalmente íntegra!

=== TODOS OS TESTES FORMAIS DO MOTOR DE INFERÊNCIA FORAM CONCLUÍDOS COM SUCESSO! ===
